In [ ]:
# imports
import pandas as pd
import pickle
import numpy as np
import krippendorff
from src.utils import load_env, load_json, get_logger

# variables
DATASET = "subframes_guns"
if DATASET == "tweets_immigration":
    DATASET_OUT_NAME = "tweets_immigration_labeled"
else:
    DATASET_OUT_NAME = DATASET
env_vars = load_env()
logger = get_logger("met-rep-eval")

In [ ]:
# load data - both annotators and gt
completed_anns_path = f"{env_vars['RESULTS_DIR']}/annotation/intrusion/completed_anns"
base_filename = f"{DATASET_OUT_NAME}_mets_only_qwen_fe_clusters_pckmeans"

ann0_df = pd.read_csv(f"{completed_anns_path}/{base_filename} - ann0.csv", dtype=str)
ann0_df["ann0_answer"] = ann0_df["answer"]
ann0_df = ann0_df[["ann0_answer"]].astype(str)

ann1_df = pd.read_csv(f"{completed_anns_path}/{base_filename} - ann1.csv", dtype=str)
ann1_df["ann1_answer"] = ann1_df["answer"]
ann1_df = ann1_df[["ann1_answer"]].astype(str)

ann0_df = ann0_df.apply(lambda col: col.str.upper() if col.dtype == "object" else col)
ann1_df = ann1_df.apply(lambda col: col.str.upper() if col.dtype == "object" else col)
ann_df = ann0_df.merge(ann1_df, left_index=True, right_index=True)

# read and format gt data
ground_truth_path = f"{env_vars['RESULTS_DIR']}/annotation/intrusion/ground_truth/{DATASET_OUT_NAME}_mets_only_qwen_fe_clusters_pckmeans_solutions.pkl"
with open(ground_truth_path, 'rb') as f:
    ground_truth_data = pickle.load(f)

gt_df = pd.DataFrame.from_dict(ground_truth_data, orient="index")
gt_df["gt_answer"] = gt_df["answer"].astype(str)
gt_df = gt_df[["gt_answer", "difficulty"]]
ann_df.head()

In [ ]:
# calculate k-alpha
reliability_data = [
    [v for v in ann_df["ann0_answer"].tolist()],
    [v for v in ann_df["ann1_answer"].tolist()],

    ]

k = krippendorff.alpha(
    reliability_data=reliability_data,
    level_of_measurement="nominal",
)
print(k)


In [ ]:
merged_df = gt_df.merge(ann_df, left_index=True, right_index=True)
overall_ann0_acc = merged_df['ann0_answer'].eq(merged_df['gt_answer']).mean()
overall_ann1_acc = merged_df['ann1_answer'].eq(merged_df['gt_answer']).mean()
overall_acc = (overall_ann1_acc + overall_ann0_acc) / 2
print(f"Overall: {overall_acc}")

In [ ]:
easy_df = merged_df[merged_df["difficulty"] == "easy"]
med_df = merged_df[merged_df["difficulty"] == "medium"]
hard_df = merged_df[merged_df["difficulty"] == "hard"]

easy_acc_ann0 = easy_df['ann0_answer'].eq(easy_df['gt_answer']).mean()
easy_acc_ann1 = easy_df['ann1_answer'].eq(easy_df['gt_answer']).mean()
easy_acc = ((easy_acc_ann0 + easy_acc_ann1)/2).round(3)
print(f"easy accuracy: {easy_acc}")

med_acc_ann0 = med_df['ann0_answer'].eq(med_df['gt_answer']).mean()
med_acc_ann1 = med_df['ann1_answer'].eq(med_df['gt_answer']).mean()
med_acc = ((med_acc_ann0 + med_acc_ann1)/2).round(3)
print(f"Med accuracy: {med_acc}")


print(f"Overall: {((easy_acc+med_acc)/2).round(2)}")
